The only diff of this version with the auto version of this code is that Start_Number is locked to 1 here

In [ ]:
# -*- coding: utf-8 -*-
"""
Clone-only extractor for Android instrumentation-testing related files.

Features:
- Auto-detect default branch via: git ls-remote --symref <repo> HEAD
- Shallow clone that branch (--depth 1)
- Extract BOTH config files AND test sources
- Save with flat filenames: {owner}.{repo}__{ci_platform}++{R|H}++{file_lower}
- Overwrite existing files if same flat name
- Split into two buckets:
    - All_Config_Files: Gradle, settings, gradle.properties, AndroidManifest.xml (any), CI YAMLs, SH, CI-ish JSON
    - All_Test_Files: everything under src/androidTest/** (kt/java/xml/etc.)
- CSV index with: owner, repo, repo_url, default_branch, commit_sha,
  relative_path, filename, flat_filename, ci_platform, ci_source, html_url,
  saved_to, bucket, components ("1", "2", "3", or combos like "1;2")
- Project metadata via GitHub API + paginated counts (contributors, pulls, commits)
- Per-commit CSV with "touches_androidTest / touches_github_workflows / touches_gradle" counters


"""

import os, re, csv, random, stat, shutil, subprocess, requests
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
from dotenv import load_dotenv, set_key

# ========= CONFIG =========
MAX_PROJECTS = 4697
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# ---- Inputs / Outputs ----
csv_path = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8\URL_List.csv")
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8")

clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
config_bucket = base_dir / "All_Config_Files"
tests_bucket = base_dir / "All_Test_Files"
commits_dir = base_dir / "Commits"
git_metadata_dir = base_dir / "Git_Metadata"

# Global CSV index
flat_index_csv = config_bucket.parent / "All_Config_Index.csv"

metadata_path = base_dir / "Project_Metadata.csv"
list_of_config_path = base_dir / "List_of_Config.csv"

# ========= ENV / TOKENS =========
load_dotenv(ENV_FILE)
TOKENS = [os.getenv(f'GITHUB_TOKEN_{i}') for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]
if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_tokens.env (API metadata might be rate limited).")
token_index = 0

START_NUMBER = 1
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# ========= Ensure legacy folders =========
for path in [clone_dir, commits_dir, cloned_sample_dir, git_metadata_dir, config_bucket, tests_bucket]:
    path.mkdir(parents=True, exist_ok=True)

# ========= CI platform patterns =========
ci_patterns = {
    r'\.travis\.ya?ml$': 'Travis_CI',
    r'\.appveyor\.ya?ml$': 'AppVeyor',
    r'appveyor\.ya?ml$': 'AppVeyor',
    r'circle\.ya?ml$': 'Circle_CI',
    r'\.circleci/config\.(yml|yaml)$': 'Circle_CI',   # already both, good
    r'azure-pipelines\.ya?ml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',  # already both
    r'bitbucket-pipelines\.ya?ml$': 'Bitbucket',
    r'\.gitlab-ci\.ya?ml$': 'GitLab',
    r'Jenkinsfile\.ya?ml$': 'Jenkins',
    r'bitrise\.ya?ml$': 'Bitrise',
    r'bamboo\.ya?ml$': 'Bamboo',
    r'codeship-services\.ya?ml$': 'Codeship',
    r'\.gocd\.ya?ml$': 'GoCD',
    r'\.cirrus\.ya?ml$': 'Cirrus',
    r'wercker\.ya?ml$': 'Wercker',
    r'semaphore\.ya?ml$': 'Semaphore',
    r'codemagic\.ya?ml$': 'Nevercode',
}



# ========= Inclusion patterns =========
LIKELY_CI_JSON = {"android-studio-loading.json", "saucectl.config.json", "firebase.json", "test-lab.json"}
INCLUDE_PATTERNS = (
    # Gradle, settings, properties
    re.compile(r'(?:^|.*/)build\.gradle(\.kts)?$', re.I),
    re.compile(r'(?:^|.*/)settings\.gradle(\.kts)?$', re.I),
    re.compile(r'(?:^|.*/)gradle\.properties$', re.I),

    # Android manifests (any source set)
    re.compile(r'(?:^|.*/)AndroidManifest\.xml$', re.I),

    # androidTest tree (any file)
    re.compile(r'(?:^|.*/)src/androidTest/.*', re.I),

    # Shell scripts anywhere
    re.compile(r'(?:^|.*/)[^/]+\.sh$', re.I),

    # JSON (filtered later unless under workflows)
    re.compile(r'(?:^|.*/)[^/]+\.json$', re.I),

    # ALLOW ANY YAML ANYWHERE so we can tag H vs R later
    re.compile(r'(?:^|.*/)[^/]+\.(yml|yaml)$', re.I),
    # JenkinsFiles
    re.compile(r'(?:^|.*/)Jenkinsfile$', re.I)
)

# ========= Trigger & Env keyword detectors =========
TRIGGER_REGEX = re.compile(
    r'(?:^|\s)(?:\.\/)?gradle(?:w)?\s+.*connected(?:Android|Debug|Release)?AndroidTest'
    r'|connectedCheck\b|adb\s+shell\s+am\s+instrument\b|gcloud\s+firebase\s+test\s+android\s+run\b',
    re.I
)
EXEC_ENV_REGEX = re.compile(
    r'reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|'
    r'\bsdkmanager\b|\bavdmanager\b|\bemulator\b|'
    r'browserstack/|appcenter\s+test\s+run|saucectl|'
    r'gradle\s+managed\s+devices|managedDevices|managedVirtualDevice|cleanManagedDevices',
    re.I
)

# ========= CSV setup =========
if not flat_index_csv.exists():
    with open(flat_index_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "owner","repo","repo_url","default_branch","commit_sha",
            "relative_path","filename","flat_filename","ci_platform","ci_source","html_url",
            "saved_to","bucket","components"
        ])
        writer.writeheader()

# ========= Helpers =========
def run(cmd, cwd=None, check=True):
    return subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=check)

def strict_yaml_match(rel_path: str):
    """
    Returns (is_strict: bool, platform: str) for YAML files only.
    is_strict=True if rel_path matches a strict CI location from ci_patterns.
    platform is the mapped platform when strict, otherwise 'CI_YAML'.
    """
    p = rel_path.replace("\\", "/")
    # check against specific patterns to identify platform
    for pattern, platform in ci_patterns.items():
        if re.search(pattern, p, re.IGNORECASE):
            return True, platform
    return False, "CI_YAML"

def is_test_like(rel_path: str, filename: str) -> bool:
    rp = rel_path.replace("\\", "/").lower()
    fn = filename.lower()

    # obvious directories
    if "/src/" in rp and ("/test/" in rp or "/androidtest/" in rp or "/uitest/" in rp):
        return True

    # filenames that look like tests
    if re.search(r'(?:^|/).*(?:test|tests|androidtest|uitest|instrumented|instrumentation)\.(?:kt|java)$', rp, re.I):
        return True

    # xml configs that look test-related (heuristics)
    if fn.endswith(".xml") and (
        "test" in fn or "orchestrator" in fn or "runner" in fn or
        "espresso" in fn or "uiautomator" in fn
    ):
        return True

    return False

def parse_owner_repo(url: str):
    parts = urlparse(url)
    if parts.netloc.lower() != "github.com":
        raise ValueError("Only github.com URLs supported")
    pieces = parts.path.strip("/").split("/")
    if len(pieces) < 2:
        raise ValueError("Invalid GitHub URL")
    owner = pieces[0]
    repo = pieces[1].replace(".git", "")
    return owner, repo

def detect_default_branch_via_git(repo_url: str) -> str:
    p = run(["git", "ls-remote", "--symref", repo_url, "HEAD"])
    for line in p.stdout.splitlines():
        s = line.strip()
        if s.startswith("ref: ") and s.endswith("HEAD"):
            ref = s.split()[1]
            if ref.startswith("refs/heads/"):
                return ref.split("/", 2)[2]
    for guess in ("main", "master"):
        try:
            run(["git", "ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            pass
    raise RuntimeError("Could not determine default branch (ls-remote)")

def shallow_clone_branch(repo_url: str, dest: Path, branch: str):
    dest.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch, repo_url, str(dest)])

def matches_path(rel_path: str, filename: str) -> bool:
    rp = rel_path.replace("\\", "/")
    rp_lower = rp.lower()
    for pat in INCLUDE_PATTERNS:
        if pat.search(rp):  # .search() (not .match())
            # JSON filtering unless under workflows
            if filename.lower().endswith(".json"):
                name_only = Path(rel_path).name.lower()
                if (name_only not in LIKELY_CI_JSON) and (not rp_lower.startswith(".github/workflows/")):
                    return False
            return True
    return False

def detect_ci_platform_with_source(rel_path: str, filename: str):
    """
    Returns (ci_platform, source) where source is:
      'R' = strict regex (from ci_patterns)
      'H' = heuristic fallback
    Used for NON-YAML files in this script (YAMLs are handled separately).
    """
    p = rel_path.replace("\\", "/")
    f = filename.lower()
    pl = p.lower()

    # 1) Strict pass – if a non-yaml file happens to live under a strict location, keep it aligned
    for pattern, platform in ci_patterns.items():
        if re.search(pattern, p, re.IGNORECASE):
            return platform, "R"

    # 2) Heuristic pass – names aligned to ci_patterns
    if pl.startswith(".github/workflows/"):               return "GitHub_Actions", "H"
    if "/.circleci/" in pl or "circle.yml" in pl:         return "Circle_CI", "H"
    if "azure" in pl or f == "azure-pipelines.yml":       return "Azure_Pipelines", "H"
    if "/.gitlab/" in pl or f == ".gitlab-ci.yml":        return "GitLab", "H"
    if "bitrise" in pl or "bitrise" in f:                 return "Bitrise", "H"
    if "jenkins" in pl or f == "jenkinsfile" or f == "jenkinsfile.yml":
                                                           return "Jenkins", "H"
    if "bitbucket-pipelines" in pl:                       return "Bitbucket", "H"
    if "semaphore" in pl:                                 return "Semaphore", "H"
    if "wercker" in pl:                                   return "Wercker", "H"
    if "codeship" in pl:                                  return "Codeship", "H"
    if "cirrus" in pl:                                    return "Cirrus", "H"
    if ".gocd" in pl or "gocd" in pl:                     return "GoCD", "H"
    if "bamboo" in pl:                                    return "Bamboo", "H"
    if "codemagic" in pl:                                 return "Nevercode", "H"
    if "appveyor" in pl or f in {"appveyor.yml", ".appveyor.yml"}:
                                                           return "AppVeyor", "H"
    if "travis" in pl or f == ".travis.yml":              return "Travis_CI", "H"

    # Generic fallbacks (non-yaml types may still be CI-related)
    if f.endswith(".sh"):                                 return "Shell", "H"
    if f.endswith(".gradle") or f.endswith(".kts"):       return "Gradle", "H"
    if f == "androidmanifest.xml":                        return "Manifest", "H"
    if "src/androidtest/" in pl:                          return "AndroidTest", "H"

    return "Other", "H"


def classify_components(rel_path: str, filename: str, content: str) -> str:
    rp = rel_path.replace("\\", "/").lower()
    fn = filename.lower()
    comps = set()

    # Component 1
    if fn.endswith((".gradle", ".gradle.kts",)) or \
       fn in {"androidmanifest.xml"} or \
       fn in {"gradle.properties", "settings.gradle", "settings.gradle.kts"} or \
       "/src/androidtest/" in rp:
        comps.add("1")

    # Component 2
    if EXEC_ENV_REGEX.search(content or "") or \
       fn.endswith((".yml", ".yaml", ".sh")) or \
       ("manageddevices" in (content or "").lower()):
        if EXEC_ENV_REGEX.search(content or "") or fn.endswith(".sh") or "/.github/workflows/" in rp:
            comps.add("2")

    # Component 3
    if TRIGGER_REGEX.search(content or ""):
        comps.add("3")

    if not comps:
        if fn.endswith((".yml", ".yaml", ".sh")):
            comps.add("2")
        elif "/src/androidtest/" in rp:
            comps.add("1")

    return ";".join(sorted(comps)) if comps else ""

def get_count(api_url, headers):
    per_page = 100
    page = 1
    total_items = 0
    try:
        while True:
            response = requests.get(api_url, headers=headers, params={"per_page": per_page, "page": page}, timeout=30)
            if response.status_code != 200:
                break
            items = response.json()
            if not isinstance(items, list):
                break
            total_items += len(items)
            if len(items) < per_page:
                break
            page += 1
    except Exception:
        pass
    return total_items

def extract_commit_metadata(repo_path: Path, output_folder: Path):
    try:
        cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
        result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
        commit_hashes = result_hashes.stdout.strip().split("\n")

        rows = []
        for commit in commit_hashes:
            cmd_metadata = ["git", "-C", str(repo_path), "show", "--quiet",
                            f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit]
            result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True)
            if not result_metadata.stdout:
                continue
            parts = result_metadata.stdout.strip().split("|", maxsplit=4)
            if len(parts) < 5:
                continue

            cmd_files = ["git", "-C", str(repo_path), "show", "--name-only", "--pretty=format:", commit]
            result_files = subprocess.run(cmd_files, capture_output=True, text=True, check=True)
            changed_files = [f.strip() for f in result_files.stdout.strip().split("\n") if f.strip()]

            # normalize case
            lower_changed = [c.lower() for c in changed_files]
            count_androidTest = sum("androidtest" in c for c in lower_changed)
            count_github_workflows = sum(".github/workflows" in c for c in lower_changed)
            count_gradle = sum("build.gradle" in c for c in lower_changed)

            rows.append({
                "commit_hash": parts[0],
                "author_name": parts[1],
                "author_email": parts[2],
                "commit_date": parts[3],
                "commit_message": parts[4],
                "touches_androidTest": count_androidTest > 0,
                "count_androidTest": count_androidTest,
                "touches_github_workflows": count_github_workflows > 0,
                "count_github_workflows": count_github_workflows,
                "touches_gradle": count_gradle > 0,
                "count_gradle": count_gradle
            })

        if rows:
            dfc = pd.DataFrame(rows)
            output_folder.mkdir(parents=True, exist_ok=True)
            flat_filename = f"{repo_path.name}__GitMetadata++contributors_commits.csv"
            dfc.to_csv(output_folder / flat_filename, index=False)
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to extract commit data for {repo_path.name}: {e}")

# ========= Load URL list =========
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# ========= Sampling =========
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# ========= Process =========
review_status_rows = []
CLONE_FAILURE_COLUMNS = ["repo_index", "repo_name", "github_url", "error_message"]

for i in range(START_NUMBER - 1, min(len(df), MAX_PROJECTS)):
    url = df.iloc[i]['github_url']
    owner_repo = urlparse(url).path.strip("/").split("/")
    if len(owner_repo) < 2:
        continue
    owner, project = owner_repo[0], owner_repo[1].replace(".git", "")
    repo_index = str(i).zfill(4)
    repo_name_tag = f"{repo_index}.{owner}.{project}"
    repo_path = clone_dir / repo_name_tag
    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name_tag}...")

    # Clone default branch only
    try:
        default_branch = detect_default_branch_via_git(url)
        print(f"📌 Default branch: {default_branch}")
        shallow_clone_branch(url, repo_path, default_branch)
        print("✅ Clone complete")
    except Exception as e:
        error_message = (str(e) or "Unknown error").strip()
        print(f"❌ Clone failed for {repo_name_tag}\n{error_message}")
        review_status_rows.append({"html_url": url.strip(), "clone_status": "no", "yml_detected": "no"})
        pd.DataFrame([review_status_rows[-1]]).to_csv(
            base_dir / "Clone_Status.csv", mode='a', header=not (base_dir / "Clone_Status.csv").exists(), index=False
        )
        fail_row = {"repo_index": repo_index, "repo_name": repo_name_tag, "github_url": url.strip(), "error_message": error_message}
        fail_path = base_dir / "Clone_Failures.csv"
        pd.DataFrame([fail_row])[CLONE_FAILURE_COLUMNS].to_csv(
            fail_path, mode='a', header=not fail_path.exists(), index=False
        )
        continue

    # Commit count + SHA + commit metadata with touches counters
    try:
        local_commit_count = int(subprocess.run(['git', '-C', str(repo_path), 'rev-list', '--count', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    except subprocess.CalledProcessError:
        local_commit_count = 0
        print(f"⚠️ Could not get commit count for {repo_name_tag}")

    head_sha = ""
    try:
        head_sha = subprocess.run(['git', '-C', str(repo_path), 'rev-parse', '--verify', 'HEAD'],
                                  capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError:
        pass

    if local_commit_count > 0:
        extract_commit_metadata(repo_path, git_metadata_dir)

    # Walk files and extract
    any_yml = False
    legacy_config_rows = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            if not matches_path(rel_path, file):
                continue

            # Read content for component classification
            try:
                content = file_path.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                content = ""

            is_yaml = file_lower.endswith(('.yml', '.yaml'))

            # Determine CI platform + source
            if is_yaml:
                is_strict, platform = strict_yaml_match(rel_path)
                if is_strict:
                    ci_source = "R"
                    ci_platform = platform
                else:
                    ci_source = "H"
                    ci_platform = "CI_YAML"
            else:
                ci_platform, ci_source = detect_ci_platform_with_source(rel_path, file)

            # Determine bucket (tests vs config)
            is_test_source = is_test_like(rel_path, file)
            bucket = "All_Test_Files" if is_test_source else "All_Config_Files"

            # Filename with marker matching ci_source
            flat_filename = f"{owner}.{project}__{ci_platform}++{ci_source}++{file_lower}"

            dest_path = (tests_bucket if is_test_source else config_bucket) / flat_filename
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file_path, dest_path)  # overwrite

            if is_yaml:
                any_yml = True

            # Component classification
            components = classify_components(rel_path, file_lower, content)

            # Write unified CSV index
            with open(flat_index_csv, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=[
                    "owner","repo","repo_url","default_branch","commit_sha",
                    "relative_path","filename","flat_filename","ci_platform","ci_source","html_url",
                    "saved_to","bucket","components"
                ])
                writer.writerow({
                    "owner": owner,
                    "repo": project,
                    "repo_url": url.strip(),
                    "default_branch": default_branch,
                    "commit_sha": head_sha,
                    "relative_path": rel_path,
                    "filename": file,
                    "flat_filename": flat_filename,
                    "ci_platform": ci_platform,
                    "ci_source": ci_source,
                    "html_url": f"https://github.com/{owner}/{project}/blob/{default_branch}/{rel_path}",
                    "saved_to": str(dest_path),
                    "bucket": bucket,
                    "components": components
                })

            # Maintain legacy List_of_Config.csv
            legacy_config_rows.append({
                "html_url": url.strip().rstrip('/'),
                "repo_name": repo_name_tag,
                "config_file_path": flat_filename,
                "original_rel_path": rel_path,
                "file_name": file,
                "file_type": file_lower.split('.')[-1]
            })

    # Clone status
    review_status_row = {"html_url": url.strip(), "clone_status": "yes", "yml_detected": "yes" if any_yml else "no"}
    pd.DataFrame([review_status_row]).to_csv(
        base_dir / "Clone_Status.csv", mode='a', header=not (base_dir / "Clone_Status.csv").exists(), index=False
    )

    # Persist legacy List_of_Config.csv
    if legacy_config_rows:
        ldf = pd.DataFrame(legacy_config_rows)
        if list_of_config_path.exists():
            ldf.to_csv(list_of_config_path, mode='a', header=False, index=False)
        else:
            ldf.to_csv(list_of_config_path, mode='w', header=True, index=False)

    # Project metadata + paginated counts + contributors
    try:
        headers = {}
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        base_api = f"https://api.github.com/repos/{owner}/{project}"
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json() if r.status_code == 200 else {}

        # Paginated counts (rotate tokens)
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        contributors_count = get_count(f"{base_api}/contributors", headers)

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        pulls_count = get_count(f"{base_api}/pulls?state=all", headers)

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        commits_count = get_count(f"{base_api}/commits", headers)

        metadata_row = {
            "html_url": url,
            "repo_index": repo_index,
            "repo_name": repo_name_tag,
            "id": data.get("id"),
            "name": data.get("name"),
            "full_name": data.get("full_name"),
            "owner": data.get("owner", {}).get("login") if data.get("owner") else None,
            "private": data.get("private"),
            "fork": data.get("fork"),
            "created_at": data.get("created_at"),
            "updated_at": data.get("updated_at"),
            "pushed_at": data.get("pushed_at"),
            "homepage": data.get("homepage"),
            "size": data.get("size"),
            "stargazers_count": data.get("stargazers_count"),
            "language": data.get("language"),
            "forks_count": data.get("forks_count"),
            "open_issues_count": data.get("open_issues_count"),
            "license": data.get("license", {}).get("name") if data.get("license") else None,
            "topics": ", ".join(data.get("topics", [])) if data.get("topics") else None,
            "visibility": data.get("visibility"),
            "default_branch": data.get("default_branch"),
            "has_issues": data.get("has_issues"),
            "has_projects": data.get("has_projects"),
            "has_downloads": data.get("has_downloads"),
            "has_wiki": data.get("has_wiki"),
            "has_pages": data.get("has_pages"),
            "archived": data.get("archived"),
            "disabled": data.get("disabled"),
            "allow_forking": data.get("allow_forking"),
            "is_template": data.get("is_template"),
            "web_commit_signoff_required": data.get("web_commit_signoff_required"),
            "contributors": contributors_count,
            "pull_requests": pulls_count,
            "commits_GitAPI": commits_count,
            "local_commit_count": local_commit_count
        }
        mdf = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            mdf.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            mdf.to_csv(metadata_path, mode='w', header=True, index=False)

        # Contributors file
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        contrib_url = f"{base_api}/contributors"
        r_contrib = requests.get(contrib_url, headers=headers, timeout=30)
        if r_contrib.status_code == 200:
            contributor_logins = [c['login'] for c in r_contrib.json()]
            contributors_text = "\n".join(contributor_logins)
            contributors_filename = f"{owner}.{project}__Contributors++list.txt"
            contributors_path = commits_dir / contributors_filename
            with open(contributors_path, "w", encoding="utf-8") as f:
                f.write(contributors_text)

    except Exception as e:
        print(f"⚠️ Metadata or contributors error for {repo_name_tag}: {e}")

    # Cleanup or keep sample
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📆 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, onerror=lambda f,p,e: (os.chmod(p, stat.S_IWRITE), f(p)))
            print(f"🕵️ Deleted cloned repo: {repo_name_tag}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name_tag}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

# Final dedupes
for p in [base_dir / "List_of_Config.csv",
          base_dir / "Clone_Failures.csv",
          base_dir / "Project_Metadata.csv",
          base_dir / "Clone_Status.csv"]:
    if p.exists():
        try:
            dfp = pd.read_csv(p)
            dfp.drop_duplicates().to_csv(p, index=False)
        except Exception:
            pass

print("\n✅ Process complete.")


🔁 Loaded SAMPLE_LIST from .env with 150 indices.

🔍 [1/1] Processing 0000.AdamMc331.AndroidStudyGuide...
📌 Default branch: development
✅ Clone complete
🕵️ Deleted cloned repo: 0000.AdamMc331.AndroidStudyGuide

✅ Process complete.
